In [23]:
import pandas as pd
import numpy as np 
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor
from sklearn.calibration import LabelEncoder

In [24]:
df = pd.read_excel('Online_Store.xlsx', 
                   sheet_name='Dim_Fact')
df_date = pd.read_excel("Online_Store.xlsx", sheet_name="Dim_Date")
df_shipping = pd.read_excel("Online_Store.xlsx", sheet_name="Dim_Shipping")
df_market = pd.read_excel("Online_Store.xlsx", sheet_name="Dim_Market")
df_priority = pd.read_excel("Online_Store.xlsx", sheet_name="Dim_OrderPriority")
df_geo = pd.read_excel("Online_Store.xlsx", sheet_name="Dim_Geography")
df_customer = pd.read_excel("Online_Store.xlsx", sheet_name="Dim_Customer")
df_product = pd.read_excel("Online_Store.xlsx", sheet_name="Dim_Product")

all_dim = df.merge(df_date, left_on="Date_ID", right_on="Date_ID") \
            .merge(df_shipping, left_on="Shipping_ID", right_on="Shipping_ID") \
            .merge(df_market, left_on="Market_ID", right_on="Market_ID") \
            .merge(df_priority, left_on="OrderPriority_ID", right_on="Priority_ID") \
            .merge(df_geo, left_on="Geography_ID", right_on="Geography_ID") \
            .merge(df_customer, left_on="Customer_ID", right_on="Customer_ID") \
            .merge(df_product, left_on="Product_ID", right_on="Product_ID")

In [25]:
all_dim.columns

Index(['Fact_ID', 'Customer_ID', 'Product_ID', 'Shipping_ID',
       'OrderPriority_ID', 'Market_ID', 'Date_ID', 'Geography_ID', 'Sales',
       'Quantity', 'Discount', 'Profit', 'Shipping_Cost', 'Is_Returned',
       'Order_Date_M', 'Order_Date_D', 'Order_Date_Y', 'Shipping_Date_M',
       'Shipping_Date_D', 'Shipping_Date_Y', 'Shipping_Mode', 'Market_Place',
       'Priority_ID', 'Priority_Level', 'City', 'State', 'Country', 'Region',
       'Longitude', 'Latitude', 'Customer_Name', 'Segment', 'Product_Brand',
       'Product_Name', 'Product_Feature', 'Category', 'Sub-Category'],
      dtype='object')

In [26]:
all_dim['Order_Date'] = pd.to_datetime(
    all_dim['Order_Date_Y'].astype(int).astype(str) + '/' + 
    all_dim['Order_Date_M'].astype(int).astype(str) + '/' + 
    all_dim['Order_Date_D'].astype(int).astype(str)
)

all_dim['Shipping_Date'] = pd.to_datetime(
    all_dim['Shipping_Date_Y'].astype(int).astype(str) + '/' + 
    all_dim['Shipping_Date_M'].astype(int).astype(str) + '/' + 
    all_dim['Shipping_Date_D'].astype(int).astype(str)
)

# numerical features
all_dim['Lag_Time_Days'] = (all_dim['Shipping_Date'] - all_dim['Order_Date']).dt.days
all_dim['Discount_to_Sales_Ratio'] = all_dim['Discount'] / (all_dim['Sales'] + 1)
all_dim['Avg_Sales_per_Product'] = all_dim.groupby('Product_ID')['Sales'].transform('mean')

#cat features
all_dim['Order_Season'] = all_dim['Order_Date_M'].map({
    1: 'Winter', 2: 'Winter', 3: 'Spring', 4: 'Spring', 
    5: 'Spring', 6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Fall', 10: 'Fall', 11: 'Fall', 12: 'Winter'
})

all_dim['Is_Peak_Season'] = all_dim['Order_Date_M'].isin([11, 12]).astype(int)
all_dim['PeakSeason_x_Discount'] = all_dim['Discount'] * all_dim['Is_Peak_Season']
#new
all_dim['Shipping_Cost_per_Unit'] = all_dim['Shipping_Cost'] / (all_dim['Quantity'] + 1)
all_dim['Sales_per_Unit'] = all_dim['Sales'] / (all_dim['Quantity'] + 1)
all_dim['Discount_per_Unit'] = all_dim['Discount'] / (all_dim['Quantity'] + 1)
all_dim['Net_Sales_After_Discount'] = all_dim['Sales'] - all_dim['Discount']
all_dim['Discount_x_Quantity'] = all_dim['Discount'] * all_dim['Quantity']
all_dim['Sales_x_Quantity'] = all_dim['Sales'] * all_dim['Quantity']



In [27]:
all_dim.duplicated().sum()

0

In [28]:
null_counts = all_dim.isnull().sum()
print(null_counts[null_counts > 0])


Product_Name        232
Product_Feature    7590
dtype: int64


In [29]:
#2
numeric_features = ['Sales', 'Quantity', 'Discount', 'Shipping_Cost','Discount_to_Sales_Ratio'
    ,'Avg_Sales_per_Product','PeakSeason_x_Discount', 'Net_Sales_After_Discount',
'Shipping_Cost_per_Unit','Sales_per_Unit','Discount_x_Quantity','Sales_x_Quantity','Discount_per_Unit']


categorical_features = ['Category', 'Sub-Category', 'Market_Place', 'Priority_Level']


In [30]:
numeric_features

['Sales',
 'Quantity',
 'Discount',
 'Shipping_Cost',
 'Discount_to_Sales_Ratio',
 'Avg_Sales_per_Product',
 'PeakSeason_x_Discount',
 'Net_Sales_After_Discount',
 'Shipping_Cost_per_Unit',
 'Sales_per_Unit',
 'Discount_x_Quantity',
 'Sales_x_Quantity',
 'Discount_per_Unit']

In [31]:
encoded_features = []
label_encoders = {}

for col in categorical_features:
    if col in categorical_features:
        le = LabelEncoder()
        all_dim[col + '_encoded'] = le.fit_transform(all_dim[col].astype(str))
        label_encoders[col] = le
        encoded_features.append(col + '_encoded')

X = all_dim[numeric_features + encoded_features]
y = all_dim['Profit']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [32]:
def run_model_pipeline(model, param_grid, X_train, y_train, X_test, y_test, model_name):
    print(f"\n------------- {model_name} --------------")

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)

    print('Best parameters : ')
    print(grid_search.best_params_)
    print('----------------------------------------')

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2_test = r2_score(y_test, y_pred)

    print(f"\nEvaluating best model on test data for {model_name}:")
    print(f"MSE     : {mse:.2f}")
    print(f"MAE     : {mae:.2f}")
    print(f"The Accuracy of test set, R2 Score: {r2_test:.2f}")

    y_pred_train = best_model.predict(X_train)
    r2_train = r2_score(y_train, y_pred_train)
    print(f'The Accuracy of train set, R2 Score:  {r2_train:.2f}')

    print('')
    overfit_gap = r2_train - r2_test
    print(f'overfit_gap :  {overfit_gap:.2f}')
    if overfit_gap > 0.1:
        print('overfitting is possible')

    if hasattr(best_model, 'feature_importances_'):
        feature_importance = pd.DataFrame({
            'feature': X_train.columns,
            'importance': best_model.feature_importances_
        }).sort_values('importance', ascending=False)

        print(f"Important features for {model_name}:")
        print("-----------------------------------------")
        print(feature_importance)

    return best_model


In [33]:
# Define param grids and run each model via the function

# XGB
xgb_param_grid = {
    'n_estimators': [200],
    'max_depth': [3],         
    'learning_rate': [0.05],
    'subsample': [0.8],        
    'colsample_bytree': [0.9],
    'min_child_weight': [3, 5, 7],  
    'reg_lambda': [3],
    'reg_alpha': [0],
    'gamma': [0, 0.05, 0.1] 
}

best_xgb = run_model_pipeline(
    XGBRegressor(random_state=42, objective='reg:squarederror'),
    xgb_param_grid,
    X_train, y_train, X_test, y_test,
    model_name='XGB'
)



------------- XGB --------------
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best parameters : 
{'colsample_bytree': 0.9, 'gamma': 0, 'learning_rate': 0.05, 'max_depth': 3, 'min_child_weight': 3, 'n_estimators': 200, 'reg_alpha': 0, 'reg_lambda': 3, 'subsample': 0.8}
----------------------------------------

Evaluating best model on test data for XGB:
MSE     : 8889.80
MAE     : 37.13
The Accuracy of test set, R2 Score: 0.73
The Accuracy of train set, R2 Score:  0.78

overfit_gap :  0.05
Important features for XGB:
-----------------------------------------
                     feature  importance
2                   Discount    0.160617
0                      Sales    0.139194
7   Net_Sales_After_Discount    0.127980
9             Sales_per_Unit    0.079572
10       Discount_x_Quantity    0.070041
1                   Quantity    0.059407
11          Sales_x_Quantity    0.056053
12         Discount_per_Unit    0.051848
16    Priority_Level_encoded    0.046609
6      Pea

In [34]:
# Decision Tree
dt_param_grid = {
    'max_depth': [4, 6, 8, 10, 12],  
    'min_samples_split': [10, 20, 30, 40], 
    'min_samples_leaf': [5, 10, 15, 20]
}

best_dt = run_model_pipeline(
    DecisionTreeRegressor(random_state=42),
    dt_param_grid,
    X_train, y_train, X_test, y_test,
    model_name='Decision Tree'
)


------------- Decision Tree --------------
Fitting 5 folds for each of 80 candidates, totalling 400 fits
Best parameters : 
{'max_depth': 6, 'min_samples_leaf': 5, 'min_samples_split': 10}
----------------------------------------

Evaluating best model on test data for Decision Tree:
MSE     : 10645.24
MAE     : 38.74
The Accuracy of test set, R2 Score: 0.68
The Accuracy of train set, R2 Score:  0.74

overfit_gap :  0.05
Important features for Decision Tree:
-----------------------------------------
                     feature  importance
2                   Discount    0.382879
7   Net_Sales_After_Discount    0.362922
0                      Sales    0.199832
4    Discount_to_Sales_Ratio    0.024482
14      Sub-Category_encoded    0.012522
10       Discount_x_Quantity    0.008149
11          Sales_x_Quantity    0.005925
3              Shipping_Cost    0.003289
15      Market_Place_encoded    0.000000
13          Category_encoded    0.000000
12         Discount_per_Unit    0.000000
8 

In [35]:
# Random Forest
rf_param_grid = {
    'n_estimators': [200],
    'max_depth': [6, 8],
    'min_samples_split': [15, 20],
    'min_samples_leaf': [3, 5],
    'max_features': ['sqrt']
}

best_rf = run_model_pipeline(
    RandomForestRegressor(random_state=42),
    rf_param_grid,
    X_train, y_train, X_test, y_test,
    model_name='RF'
)


------------- RF --------------
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best parameters : 
{'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 3, 'min_samples_split': 15, 'n_estimators': 200}
----------------------------------------

Evaluating best model on test data for RF:
MSE     : 9420.84
MAE     : 36.60
The Accuracy of test set, R2 Score: 0.72
The Accuracy of train set, R2 Score:  0.76

overfit_gap :  0.04
Important features for RF:
-----------------------------------------
                     feature  importance
0                      Sales    0.170354
7   Net_Sales_After_Discount    0.152670
2                   Discount    0.123356
9             Sales_per_Unit    0.111253
11          Sales_x_Quantity    0.086247
10       Discount_x_Quantity    0.079746
12         Discount_per_Unit    0.065487
4    Discount_to_Sales_Ratio    0.063875
5      Avg_Sales_per_Product    0.057385
3              Shipping_Cost    0.036094
8     Shipping_Cost_per_Unit    0.

In [36]:
# Gradient Boosting
gb_param_grid = {
    'n_estimators': [150],
    'max_depth': [2, 3],
    'learning_rate': [0.05],
    'subsample': [0.8],
    'min_samples_split': [10, 20],
    'min_samples_leaf': [3, 5],
    'max_features': ['sqrt']
}

best_gb = run_model_pipeline(
    GradientBoostingRegressor(random_state=42),
    gb_param_grid,
    X_train, y_train, X_test, y_test,
    model_name='GB'
)


------------- GB --------------
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best parameters : 
{'learning_rate': 0.05, 'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 3, 'min_samples_split': 10, 'n_estimators': 150, 'subsample': 0.8}
----------------------------------------

Evaluating best model on test data for GB:
MSE     : 9115.62
MAE     : 39.05
The Accuracy of test set, R2 Score: 0.73
The Accuracy of train set, R2 Score:  0.79

overfit_gap :  0.06
Important features for GB:
-----------------------------------------
                     feature  importance
0                      Sales    0.222032
7   Net_Sales_After_Discount    0.167182
2                   Discount    0.123819
9             Sales_per_Unit    0.094595
11          Sales_x_Quantity    0.071048
10       Discount_x_Quantity    0.063126
12         Discount_per_Unit    0.062813
5      Avg_Sales_per_Product    0.055118
4    Discount_to_Sales_Ratio    0.046651
3              Shipping_Cost    0.